# identity-anonymizer デモ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/identity-anonymizer/blob/master/demo.ipynb)

[GHOST](https://github.com/ai-forever/ghost)による顔交換と，差し替え可能な匿名化モデル(`Anonymizer`)を組み合わせて，画像・動画中の人物の顔を，属性(性別・年齢等)や表情を保持したまま架空の人物へ匿名化するパイプラインのデモである．リポジトリ全体の詳細は [README.md](README.md) を参照．

**このnotebookで行うこと**
1. (Colab上のみ) 実行環境のセットアップ(Python 3.9 + CUDA対応ライブラリのインストール)
2. リポジトリのクローンと学習済み重みのダウンロード
3. 画像1枚に対する匿名化のデモ(ノイズレベルを変えた比較)
4. 動画1本に対する匿名化のデモ

**実行前に**: メニューの `ランタイム > ランタイムのタイプを変更` でGPU(T4など)を選択しておくこと．

## 1. Python環境のセットアップ (Colab専用)

ローカル環境で `conda activate identity-anonymizer` 済みの場合(README.mdの「セットアップ」参照)は，このセクションを実行せず「2. リポジトリの取得」から実行してよい．

本プロジェクトは `mxnet-cu112` / `insightface==0.2.1` / `torch`(CUDAビルド)という古いビルド済みwheelに依存しており，Python 3.9が必要である．そのため，[condacolab](https://github.com/conda-incubator/condacolab)を用いてColabランタイムにPython 3.9のMiniconda環境を構築する．

**次のセルを実行するとランタイムが自動的に再起動される(「セッションがクラッシュしました」という表示が出るが，condacolabの仕様上の正常な挙動である)．再起動後は，このセルより下のセルを上から順に実行すればよい(このセルを再実行する必要はない)．**

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q condacolab
    import condacolab

    condacolab.install(python_version="3.9")

## 2. リポジトリの取得と依存関係のインストール

In [ ]:
!nvidia-smi

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = "/content/identity-anonymizer"
    if not os.path.isdir(REPO_DIR):
        !git clone --recurse-submodules https://github.com/yryo1005/identity-anonymizer.git {REPO_DIR}
    os.chdir(REPO_DIR)
else:
    # このnotebookがリポジトリルートに配置されている前提
    REPO_DIR = os.getcwd()

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("REPO_DIR:", REPO_DIR)

In [ ]:
if IN_COLAB:
    # GHOSTへの最小限のパッチ適用(動画処理中の内側の進捗バー二重表示の抑制)
    PATCH_FILE = os.path.join(REPO_DIR, "patches", "0001-silence-inner-progress-bars.patch")
    !git -C {REPO_DIR}/third_party/ghost apply --check {PATCH_FILE} 2>/dev/null && \
        git -C {REPO_DIR}/third_party/ghost apply {PATCH_FILE} || \
        echo "パッチは適用済み、またはスキップされました"

    # CUDA対応ライブラリのインストール(README.md「1. Python環境」のenvironment.ymlに準拠)
    %pip install -q --extra-index-url https://download.pytorch.org/whl/cu116 \
        torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1+cu116
    %pip install -q mxnet-cu112==1.9.1 insightface==0.2.1 onnx==1.15.0 onnxruntime-gpu==1.14.0 \
        opencv-python==4.8.1.78 scikit-image==0.22.0 kornia==0.5.4 psutil

    # 重みのダウンロード(GHOST由来 + 本リポジトリのGitHub Release由来)
    !bash {REPO_DIR}/scripts/download_ghost_weights.sh
    !bash {REPO_DIR}/scripts/download_release_weights.sh

    # 本パッケージのインストール
    %pip install -q -e {REPO_DIR}

## 3. 匿名化パイプラインの構築

`FaceAnonymizerPipeline` は，GHOSTによる顔交換モデル一式(`load_ghost_models`)と，差し替え可能な匿名化モデル(`Anonymizer`，ここでは提案手法の `VAEAnonymizer`)を組み合わせる．

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_jp_font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if os.path.exists(_jp_font_path):
    fm.fontManager.addfont(_jp_font_path)
    plt.rcParams["font.family"] = "Noto Sans CJK JP"
    plt.rcParams["axes.unicode_minus"] = False

from identity_anonymizer.faceswap import load_ghost_models, FaceAnonymizerPipeline
from identity_anonymizer.anonymizers import get_anonymizer

VAE_WEIGHT_PATH = os.path.join(REPO_DIR, "weights/anonymizers/vae/vae_512_128.pt")

models = load_ghost_models()
anonymizer = get_anonymizer("vae", weight_path=VAE_WEIGHT_PATH)
pipeline = FaceAnonymizerPipeline(models, anonymizer)

## 4. 画像1枚に対する匿名化のデモ

スケール係数(`noise_level`)を変えることで，匿名性(元の顔との類似度の低さ)と属性の一貫性のトレードオフを確認する．`TARGET_IMAGE_PATH` を書き換えれば任意の画像で試せる(詳細は [notebooks/03_inference_image.ipynb](notebooks/03_inference_image.ipynb) を参照)．

In [ ]:
import cv2
from utils.inference.image_processing import crop_face

TARGET_IMAGE_PATH = os.path.join(REPO_DIR, "sample_images/beckham.jpg")  # 任意の画像パスに変更できる
NOISE_LEVELS = [0.0, 1.0, 1.25, 1.5, 2.0]

original_full = cv2.imread(TARGET_IMAGE_PATH)
original_face = crop_face(original_full, pipeline.models.app, pipeline.crop_size)[0]

fig, axes = plt.subplots(1, len(NOISE_LEVELS) + 1, figsize=(3 * (len(NOISE_LEVELS) + 1), 3))

axes[0].imshow(original_face[:, :, ::-1])
axes[0].set_title("Original")
axes[0].axis("off")

for ax, noise_level in zip(axes[1:], NOISE_LEVELS):
    face_image, _ = pipeline.anonymize_image(TARGET_IMAGE_PATH, noise_level=noise_level)
    ax.imshow(face_image[:, :, ::-1])
    ax.set_title(f"noise_level={noise_level}")
    ax.axis("off")

fig.tight_layout()
plt.show()

## 5. 動画1本に対する匿名化のデモ

`FaceAnonymizerPipeline.anonymize_video` は，動画全体で単一の匿名化後の顔ベクトルを使用するため，出力動画内で架空の人物の見た目が時間的に一貫する．`TARGET_VIDEO_PATH` を書き換えれば任意の動画で試せる(詳細は [notebooks/04_inference_video.ipynb](notebooks/04_inference_video.ipynb) を参照)．

In [ ]:
TARGET_VIDEO_PATH = os.path.join(REPO_DIR, "sample_videos/sample2.mp4")  # 任意の動画パスに変更できる
OUT_DIR = os.path.join(REPO_DIR, "outputs/videos")
NOISE_LEVEL = 1.25

os.makedirs(OUT_DIR, exist_ok=True)
name = os.path.splitext(os.path.basename(TARGET_VIDEO_PATH))[0]
out_video_path = os.path.join(OUT_DIR, f"{name}_anonymized.mp4")

pipeline.anonymize_video(TARGET_VIDEO_PATH, out_video_path, noise_level=NOISE_LEVEL, keep_audio=True)
print("saved:", out_video_path)

In [ ]:
import numpy as np
from utils.inference.video_processing import read_video


def sample_frame_indices(num_frames: int, num_samples: int = 3):
    if num_frames <= num_samples:
        return list(range(num_frames))
    return list(np.linspace(0, num_frames - 1, num_samples).astype(int))


original_frames, _ = read_video(TARGET_VIDEO_PATH)
anonymized_frames, _ = read_video(out_video_path)

n = min(len(original_frames), len(anonymized_frames))
indices = sample_frame_indices(n, 3)

fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 4))
for col, idx in enumerate(indices):
    axes[0, col].imshow(original_frames[idx][:, :, ::-1])
    axes[0, col].set_title(f"frame {idx}")
    axes[0, col].axis("off")
    axes[1, col].imshow(anonymized_frames[idx][:, :, ::-1])
    axes[1, col].axis("off")

fig.tight_layout()
plt.show()

出力動画をnotebook上でそのまま再生して確認する．

In [ ]:
from IPython.display import Video

Video(out_video_path, embed=True, width=480)

## 次のステップ

- 匿名化モデルの差し替え(`VAEAnonymizer` ⇔ `AttributeNNAnonymizer`)は [README.md](README.md#匿名化モデルの差し替え) を参照．
- データセット構築・学習・定量評価用のnotebookは [notebooks/](notebooks/) を参照(`01_make_dataset.ipynb` 〜 `06_train_attribute_nn_anonymizer.ipynb`)．
- 新しい匿名化モデルの実装方法は [document.md](document.md) を参照．